In [ ]:
LOAD = False
MODELS = None

In [ ]:
import gc, os, subprocess, sys, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files, userdata
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "accelerate", "pyyaml"], check=True)
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("no HF_TOKEN secret: downloads work, but more slowly")

import pandas as pd
import torch
import yaml
from transformers import (AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig,
                          GenerationConfig)

pd.set_option("display.max_colwidth", 120)
DRIVE_ROOT = Path("/content/drive/MyDrive/llm-persona-ei")

def find(name, candidates):
    path = next((path for path in candidates if path.exists()), None)
    if path is None and IN_COLAB:
        print(f"{name} not found: please upload it")
        path = Path(next(iter(files.upload())))
    assert path is not None, f"{name} not found in {[str(c) for c in candidates]}"
    return path


CONFIG_PATH = find("config.yaml", [Path("config.yaml"), Path("../config.yaml"), DRIVE_ROOT / "config.yaml"])
ROOT = Path(os.environ.get("PERSONA_EI_RESULTS") or (DRIVE_ROOT if IN_COLAB else CONFIG_PATH.resolve().parent))

OUT_DIR = ROOT / "comparison" / "results" / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("config   :", CONFIG_PATH.resolve())
print("device   :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())
model_keys = list(MODELS) if MODELS else [*cfg["tested"], cfg["judge"]]
unknown = [key for key in model_keys if key not in cfg["models"]]
assert not unknown, f"not in config.yaml: {unknown}"

PROBE = [{"role": "system", "content": "SYSTEM-PROBE"}, {"role": "user", "content": "USER-PROBE"},
         {"role": "assistant", "content": "REPLY-PROBE"}, {"role": "user", "content": "SECOND-PROBE"}]
TEST_CHATS = [
    [{"role": "system", "content": "You answer in one short sentence."},
     {"role": "user", "content": "What is the capital of France?"}],
    [{"role": "system", "content": "You answer in one short sentence."},
     {"role": "user", "content": "What is the capital of France?"},
     {"role": "assistant", "content": "The capital of France is Paris."},
     {"role": "user", "content": "And what is the capital of Italy?"}],
]
TEST_TOKENS = 40


def load_tokenizer(spec):
    tokenizer = AutoTokenizer.from_pretrained(spec["repo"], revision=spec["revision"], **spec.get("tokenizer_kwargs", {}))
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_weights(spec):
    options = {"revision": spec["revision"]}
    precision = spec["precision"]
    if DEVICE == "cuda":
        has_bf16 = torch.cuda.get_device_capability()[0] >= 8
        dtype = torch.bfloat16 if has_bf16 else torch.float16
        options["device_map"] = "auto"
        if precision in ("8bit", "4bit"):
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_8bit=precision == "8bit", load_in_4bit=precision == "4bit", bnb_4bit_compute_dtype=dtype)
        else:
            precision = "bf16" if has_bf16 else "fp16"
    else:
        if precision != "bf16":
            raise RuntimeError(f"{spec['repo']}: {precision} loading needs a GPU")
        dtype, precision = torch.float32, "fp32"
    options["dtype"] = dtype
    try:
        model = AutoModelForCausalLM.from_pretrained(spec["repo"], **options)
    except ValueError:
        model = AutoModelForImageTextToText.from_pretrained(spec["repo"], **options)
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    return model.eval(), precision


def stop_tokens(spec, tokenizer):
    try:
        stops = GenerationConfig.from_pretrained(spec["repo"], revision=spec["revision"]).eos_token_id
    except OSError:
        stops = None
    stops = stops if isinstance(stops, list) else [stops]
    return {token for token in [*stops, tokenizer.eos_token_id] if token is not None}


def check_template(key):
    spec = cfg["models"][key]
    tokenizer = load_tokenizer(spec)
    options = spec.get("chat_template_kwargs", {})
    text = tokenizer.apply_chat_template(PROBE, tokenize=False, add_generation_prompt=True, **options)
    places = [text.find(message["content"]) for message in PROBE]
    expected = tokenizer.apply_chat_template(PROBE, tokenize=True, add_generation_prompt=True, **options)
    expected = expected["input_ids"] if hasattr(expected, "keys") else expected
    after_reply = text[places[2] + len("REPLY-PROBE"):].lstrip() if places[2] >= 0 else ""
    reply_end = tokenizer(after_reply, add_special_tokens=False)["input_ids"][:1]
    return {
        "system_prompt_first": -1 not in places and places == sorted(places) and text.count("SYSTEM-PROBE") == 1,
        "prompt_as_model_expects": list(tokenizer(text, add_special_tokens=False)["input_ids"]) == list(expected),
        "stops_at_end_of_reply": bool(reply_end) and reply_end[0] in stop_tokens(spec, tokenizer),
    }


def try_model(key):
    spec = cfg["models"][key]
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    started = time.time()
    tokenizer = load_tokenizer(spec)
    model, precision = load_weights(spec)
    loaded = time.time()
    try:
        prompts = [tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True,
                                                 **spec.get("chat_template_kwargs", {})) for chat in TEST_CHATS]
        encoded = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            output = model.generate(**encoded, max_new_tokens=TEST_TOKENS, do_sample=False, repetition_penalty=1.0,
                                    pad_token_id=tokenizer.pad_token_id)
        new_tokens = output[:, encoded["input_ids"].shape[1]:]
        replies = [text.strip() for text in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]
        truncated = ((new_tokens != tokenizer.pad_token_id).sum(dim=1) >= TEST_TOKENS).any().item()
    finally:
        del model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return {
        "loaded_as": precision, "load_seconds": round(loaded - started, 1), "reply_seconds": round(time.time() - loaded, 1),
        "gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 1) if DEVICE == "cuda" else None,
        "test_replies_correct": "paris" in replies[0].lower() and "rome" in replies[1].lower() and not truncated,
        "test_replies": " | ".join(replies),
    }


rows = []
for key in model_keys:
    print(f"checking {key} ...", flush=True)
    row = {"model": key, "precision": cfg["models"][key]["precision"], "error": ""}
    for step in (check_template, try_model) if LOAD else (check_template,):
        try:
            row.update(step(key))
        except Exception as error:
            row["error"] += f"{step.__name__}: {type(error).__name__}: {str(error)[:200]} "
    rows.append(row)
checks = pd.DataFrame(rows)
checks = checks[[column for column in checks if column != "error"] + ["error"]]
checks.to_csv(OUT_DIR / "model_check.csv", index=False)
print("saved to", OUT_DIR / "model_check.csv")
checks

In [ ]:
if "test_replies" in checks:
    for row in checks.itertuples():
        print(f"{row.model}: {row.test_replies}")